In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import os

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print("imports done")

imports done


In [2]:
from nba_api.stats.endpoints import LeagueDashPlayerStats, PlayerIndex

stats = LeagueDashPlayerStats(season='2023-24', per_mode_detailed='PerGame')
df_test = stats.get_data_frames()[0]

print("=== LeagueDashPlayerStats columns ===")
print(df_test.columns.tolist())

time.sleep(1)

index = PlayerIndex(season='2023-24')
df_index = index.get_data_frames()[0]

print("\n=== PlayerIndex columns ===")
print(df_index.columns.tolist())

targets = ['USG_PCT', 'PLUS_MINUS', 'WEIGHT', 'FTA', 'OFF_RATING', 'DEF_RATING']
print("\n=== Key feature availability in LeagueDashPlayerStats ===")
for col in targets:
    print(f"  {col}: {'✓' if col in df_test.columns else '✗'}")


=== LeagueDashPlayerStats columns ===
['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK', 'DD2_RANK', 'TD3_RANK', 'WNBA_FANTASY_PTS_RANK', 'TEAM_COUNT']

=== PlayerIndex columns ===
['PERSON_ID', 'PLAYER_LAST_NAME', 'PLAYER_FIRST_NAME', 'PLAYER_SLUG', 'TEAM_ID', 'TEAM_SLUG', 'IS_DEFUNCT', 'TEAM_CITY', 'TEAM_NAME', 'TEAM_ABBREVIATION', 'JERSEY_NUMBER', 'POSITION',

In [3]:
def fetch_with_retry(endpoint_func, max_retries=3, **kwargs):
    for attempt in range(max_retries):
        try:
            result = endpoint_func(**kwargs)
            time.sleep(0.6)
            return result
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(2)
    return None

def parse_height_to_inches(h):
    try:
        ft, inch = h.split('-')
        return int(ft)*12 + int(inch)
    except:
        return np.nan

def standardize_position(pos):
    if pd.isna(pos) or pos == '':
        return 'Unknown'
    pos = str(pos).upper()
    if 'G' in pos: return 'Guard'
    if 'F' in pos: return 'Forward'
    if 'C' in pos: return 'Center'
    return 'Unknown'

seasons = {
    'Modern':    ['2020-21','2021-22','2022-23','2023-24','2024-25'],
    'Three_PT':  ['2013-14','2014-15','2015-16','2016-17','2017-18'],
    'Early_00s': ['1999-00','2000-01','2001-02','2002-03','2003-04'],
}

all_dfs = []

for era, season_list in seasons.items():
    for season in season_list:
        print(f"Fetching {era} {season}...")
        res = fetch_with_retry(LeagueDashPlayerStats, season=season, per_mode_detailed='PerGame')
        if res is None:
            print(f"  FAILED: {season}")
            continue
        df_stats = res.get_data_frames()[0]
        df_stats['SEASON'] = season
        df_stats['ERA'] = era
        all_dfs.append(df_stats)
        print(f"  OK: {len(df_stats)} players")

df_raw = pd.concat(all_dfs, ignore_index=True)
print(f"\nTotal rows: {len(df_raw)}")
print(df_raw[['PLAYER_NAME','SEASON','ERA','GP','MIN']].head())


Fetching Modern 2020-21...
  OK: 540 players
Fetching Modern 2021-22...
  OK: 605 players
Fetching Modern 2022-23...
  OK: 539 players
Fetching Modern 2023-24...
  OK: 572 players
Fetching Modern 2024-25...
  OK: 569 players
Fetching Three_PT 2013-14...
  OK: 482 players
Fetching Three_PT 2014-15...
  OK: 492 players
Fetching Three_PT 2015-16...
  OK: 476 players
Fetching Three_PT 2016-17...
  OK: 486 players
Fetching Three_PT 2017-18...
  OK: 540 players
Fetching Early_00s 1999-00...
  OK: 439 players
Fetching Early_00s 2000-01...
  OK: 441 players
Fetching Early_00s 2001-02...
  OK: 440 players
Fetching Early_00s 2002-03...
  OK: 428 players
Fetching Early_00s 2003-04...
  OK: 442 players

Total rows: 7491
     PLAYER_NAME   SEASON     ERA  GP   MIN
0   Aaron Gordon  2020-21  Modern  50  27.7
1  Aaron Holiday  2020-21  Modern  66  17.8
2  Aaron Nesmith  2020-21  Modern  46  14.5
3    Abdel Nader  2020-21  Modern  24  14.8
4    Adam Mokoka  2020-21  Modern  14   4.0


In [4]:
# Already run — df_hw cached in data/raw/nba_player_season_raw.csv

# from nba_api.stats.endpoints import CommonPlayerInfo

# player_ids = df_raw['PLAYER_ID'].unique()
# print(f"Fetching info for {len(player_ids)} players (~{len(player_ids)*0.4//60:.0f} mins)...")

# height_weight = {}

# for i, pid in enumerate(player_ids):
#     if i % 50 == 0:
#         print(f"  {i}/{len(player_ids)}...")
#     try:
#         info = CommonPlayerInfo(player_id=pid)
#         row = info.get_data_frames()[0].iloc[0]
#         height_weight[pid] = {
#             'HEIGHT':   row.get('HEIGHT', np.nan),
#             'WEIGHT':   row.get('WEIGHT', np.nan),
#             'POSITION': row.get('POSITION', np.nan)
#         }
#         time.sleep(0.4)
#     except:
#         height_weight[pid] = {'HEIGHT': np.nan, 'WEIGHT': np.nan, 'POSITION': np.nan}

# df_hw = pd.DataFrame.from_dict(height_weight, orient='index')
# df_hw.index.name = 'PLAYER_ID'
# df_hw = df_hw.reset_index()
# print(f"Done. Missing HEIGHT: {df_hw['HEIGHT'].isna().sum()}/{len(df_hw)}")


Fetching info for 2195 players (~14 mins)...
  0/2195...
  50/2195...
  100/2195...
  150/2195...
  200/2195...
  250/2195...
  300/2195...
  350/2195...
  400/2195...
  450/2195...
  500/2195...
  550/2195...
  600/2195...
  650/2195...
  700/2195...
  750/2195...
  800/2195...
  850/2195...
  900/2195...
  950/2195...
  1000/2195...
  1050/2195...
  1100/2195...
  1150/2195...
  1200/2195...
  1250/2195...
  1300/2195...
  1350/2195...
  1400/2195...
  1450/2195...
  1500/2195...
  1550/2195...
  1600/2195...
  1650/2195...
  1700/2195...
  1750/2195...
  1800/2195...
  1850/2195...
  1900/2195...
  1950/2195...
  2000/2195...
  2050/2195...
  2100/2195...
  2150/2195...
Done. Missing HEIGHT: 0/2195


In [5]:
# df_merged = df_raw.merge(df_hw, on='PLAYER_ID', how='left')

# df = df_merged.copy()

# df['HEIGHT_IN'] = df['HEIGHT'].apply(parse_height_to_inches)
# df['WEIGHT']    = pd.to_numeric(df['WEIGHT'], errors='coerce')
# df['MPG']       = df['MIN']  # already per-game
# df['FG3A_RATE'] = df['FG3A'] / df['FGA'].replace(0, np.nan)
# df['FTA_RATE']  = df['FTA']  / df['FGA'].replace(0, np.nan)

# df['TEAM_GAMES']      = df.groupby('SEASON')['GP'].transform('max')
# df['MISSED_GAMES']    = df['TEAM_GAMES'] - df['GP']
# df['MISSED_GAME_RATE']= df['MISSED_GAMES'] / df['TEAM_GAMES']

# df['POSITION_GROUP'] = df['POSITION'].apply(standardize_position)

# # Quality filters
# df = df[(df['GP'] >= 10) & (df['MPG'] >= 5)].copy()
# df = df[(df['HEIGHT_IN'].between(65, 90)) | df['HEIGHT_IN'].isna()]
# df = df[(df['WEIGHT'].between(100, 350))  | df['WEIGHT'].isna()]

# print(f"Final shape: {df.shape}")
# print("\nMissingness summary:")
# cols = ['HEIGHT_IN','WEIGHT','AGE','MPG','FG3A_RATE','FTA_RATE','PLUS_MINUS','MISSED_GAME_RATE','POSITION_GROUP']
# print(df[cols].isna().sum())


Final shape: (6567, 80)

Missingness summary:
HEIGHT_IN           95
WEIGHT              96
AGE                  0
MPG                  0
FG3A_RATE            0
FTA_RATE             0
PLUS_MINUS           0
MISSED_GAME_RATE     0
POSITION_GROUP       0
dtype: int64


In [6]:
# os.makedirs('data/raw', exist_ok=True)
# os.makedirs('data/processed', exist_ok=True)

# df_merged.to_csv('data/raw/nba_player_season_raw.csv', index=False)
# df.to_csv('data/processed/nba_model_data.csv', index=False)

# print(f"Raw saved:       data/raw/nba_player_season_raw.csv  {df_merged.shape}")
# print(f"Processed saved: data/processed/nba_model_data.csv   {df.shape}")


Raw saved:       data/raw/nba_player_season_raw.csv  (7491, 72)
Processed saved: data/processed/nba_model_data.csv   (6567, 80)
